# 01 — Nonlinear Elements

**Topic**: All 5 element types — force curves, Jacobian plots, `eval()` walkthrough.

**Reference**: Krack & Gross (2019) Appendix C, Table C.1

**Estimated runtime**: < 5 seconds

## Theory

The general equation of motion with a nonlinear force term is (K&G eq. 2.1):

$$M\ddot{q} + D\dot{q} + Kq + f_{nl}(q, \dot{q}) = f_{ext}(t) \tag{1}$$

Each nonlinear element contributes a scalar force $f$ and its Jacobians $\partial f/\partial q$ and $\partial f/\partial \dot{q}$. The 5 element types from K&G Appendix C, Table C.1:

| Element | Force $f$ | $\partial f/\partial q_i$ | $\partial f/\partial \dot{q}_i$ |
|---------|-----------|--------------------------|----------------------------------|
| Cubic spring | $k_3 q_i^3$ | $3k_3 q_i^2$ | $0$ |
| Quadratic damper | $c_2 \dot{q}_i|\dot{q}_i|$ | $0$ | $2c_2|\dot{q}_i|$ |
| Tanh dry friction | $f_0\tanh(c\dot{q}_i)$ | $0$ | $f_0 c\,\text{sech}^2(c\dot{q}_i)$ |
| Unilateral spring | $k\max(q_i-\delta,0)$ | $k\,[q_i>\delta]$ | $0$ |
| Polynomial stiffness | $\sum_m c_m \prod_l q_{i_l}^{e_{m,l}}$ | (full Jacobian) | $0$ |

Each factory function returns a `NonlinearElement` with an `eval(q, dq)` method returning `(f, df_dq, df_ddq)`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt

from nlvib.nonlinearities.elements import (
    cubic_spring,
    quadratic_damper,
    tanh_dry_friction,
    unilateral_spring,
    polynomial_stiffness,
)

# Create all 5 elements
k3   = 1.0   # ← try changing this (cubic stiffness coefficient)
c2   = 0.5   # ← try changing this (quadratic damping coefficient)
f0   = 1.0   # ← try changing this (friction saturation force)
c_fr = 10.0  # ← try changing this (friction sharpness; large = Coulomb)
k_u  = 3.0   # ← try changing this (unilateral spring stiffness)
gap  = 0.3   # ← try changing this (contact gap)

el_cubic   = cubic_spring(k3=k3, dof_index=0)
el_qdamp   = quadratic_damper(c2=c2, dof_index=0)
el_tanh    = tanh_dry_friction(f0=f0, c=c_fr, dof_index=0)
el_uni     = unilateral_spring(k=k_u, gap=gap, dof_index=0)
el_poly    = polynomial_stiffness(
    exponents=np.array([[3], [5]]),
    coefficients=np.array([0.5, 0.1]),
    dof_indices=np.array([0]),
)

print("Elements created:")
for el in [el_cubic, el_qdamp, el_tanh, el_uni, el_poly]:
    print(f"  {el.label}")

In [ ]:
# Plot force curves for all 5 elements
q_range  = np.linspace(-1.0, 1.0, 300)
dq_range = np.linspace(-1.0, 1.0, 300)
q_vec    = np.zeros(1)
dq_vec   = np.zeros(1)

titles  = ['Cubic spring (vs q)', 'Quadratic damper (vs dq)',
           'Tanh friction (vs dq)', 'Unilateral spring (vs q)',
           'Polynomial stiffness (vs q)']
elements = [el_cubic, el_qdamp, el_tanh, el_uni, el_poly]
x_ranges = [q_range, dq_range, dq_range, q_range, q_range]
use_dq   = [False, True, True, False, False]

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, el, title, xr, is_dq in zip(axes, elements, titles, x_ranges, use_dq):
    forces = []
    for x in xr:
        if is_dq:
            q_vec[0] = 0.0; dq_vec[0] = x
        else:
            q_vec[0] = x;   dq_vec[0] = 0.0
        f, _, _ = el.eval(q_vec, dq_vec)
        forces.append(f)
    ax.plot(xr, forces, lw=2, color='tab:blue')
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.axvline(0, color='k', lw=0.5, ls='--')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('$\\dot{q}$' if is_dq else '$q$')
    ax.set_ylabel('$f_{nl}$')
    ax.grid(True, alpha=0.3)

fig.suptitle('Force curves — all 5 nonlinear element types (K&G App. C, Table C.1)', y=1.02)
fig.tight_layout()
fig

In [ ]:
# Compare analytic Jacobian df/dq vs. finite differences for cubic and tanh
# This verifies the Jacobian implementation is correct (K&G App. C)

def fd_jacobian(el, q0, dq0, h=1e-6):
    """First-order finite difference approximation of df/dq."""
    n = len(q0)
    jac = np.zeros(n)
    for i in range(n):
        qp = q0.copy(); qp[i] += h
        qm = q0.copy(); qm[i] -= h
        fp, _, _ = el.eval(qp, dq0)
        fm, _, _ = el.eval(qm, dq0)
        jac[i] = (fp - fm) / (2 * h)
    return jac

q_test_pts = np.array([-0.8, -0.4, 0.0, 0.4, 0.8])
dq_zero = np.zeros(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

for ax, el, name in [(ax1, el_cubic, 'Cubic spring'), (ax2, el_tanh, 'Tanh friction')]:
    analytic_jac = []
    fd_jac       = []
    for qv in q_test_pts:
        q_v = np.array([qv])
        _, df_dq_analytic, _ = el.eval(q_v, dq_zero)
        df_dq_fd = fd_jacobian(el, q_v, dq_zero)
        analytic_jac.append(df_dq_analytic[0])
        fd_jac.append(df_dq_fd[0])
    ax.plot(q_test_pts, analytic_jac, 'o-', label='Analytic $\\partial f/\\partial q$', lw=2)
    ax.plot(q_test_pts, fd_jac,       's--', label='Finite difference', alpha=0.7)
    ax.set_title(f'{name} — Jacobian check')
    ax.set_xlabel('$q$')
    ax.set_ylabel('$\\partial f / \\partial q$')
    ax.legend()
    ax.grid(True, alpha=0.3)
    max_err = max(abs(a - b) for a, b in zip(analytic_jac, fd_jac))
    print(f"{name}: max |analytic - FD| = {max_err:.2e}")

fig.tight_layout()
fig

## Try it yourself

- Change `k3 = 1.0` to `k3 = 0.0` — the cubic spring force curve becomes flat (zero force everywhere).
- Change `gap = 0.3` to `gap = -0.2` — the unilateral spring is always active (pre-loaded contact).
- Change `c_fr = 10.0` to `c_fr = 100.0` — the tanh friction approximates ideal Coulomb friction.

## Key Takeaways

- All 5 elements expose a unified `eval(q, dq) → (f, df_dq, df_ddq)` interface — no special cases in the solver.
- Analytic Jacobians match finite differences to machine precision — critical for Newton convergence in HB.
- The `polynomial_stiffness` element is the most general: any monomial in any combination of DOFs.